# SurfaceTarget-CRC — phase 05 — patch 5 6a

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.6A
#
# CORRECTED RESEARCH PANEL AND VALIDATION ROADMAP
#
# Fix:
# Robust cross-CSV scientific data integrity verification
#
# Objectives:
# 1. Validate Phase 5.4 and 5.5 source files
# 2. Compare evidence without false CSV precision failures
# 3. Detect and report genuine data discrepancies
# 4. Verify three-scenario ranking agreement
# 5. Construct the 15-candidate research panel
# 6. Preserve all 948 original candidates
# 7. Create experimental validation roadmap
# 8. Save normal-tissue evidence and research report
# 9. Save provenance and create checkpoint
#
# IMPORTANT:
# - No original data overwritten
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No candidate exclusions
# - No experimental validation claimed
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    RESULTS,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(parents=True, exist_ok=True)


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

ALL_TISSUE_PATH = (
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv"
)

THREE_WAY_PATH = (
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

P54_PROVENANCE = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

P55_PROVENANCE = (
    META /
    "surfacetarget_phase5_robustness_provenance.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.6A")
print("CORRECTED RESEARCH PANEL AND VALIDATION ROADMAP")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_files = {
    "Phase 5.4 ranking": RANKING_PATH,
    "Phase 5.5 tissue audit": ALL_TISSUE_PATH,
    "Three-scenario candidates": THREE_WAY_PATH,
    "Scenario correlations": CORRELATION_PATH,
    "Phase 5.4 provenance": P54_PROVENANCE,
    "Phase 5.5 provenance": P55_PROVENANCE
}


for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(P54_PROVENANCE, "r") as handle:
    p54 = json.load(handle)


with open(P55_PROVENANCE, "r") as handle:
    p55 = json.load(handle)


ranking_hash = sha256_file(RANKING_PATH)


if ranking_hash != p54.get("output_sha256"):

    raise RuntimeError(
        "Phase 5.4 ranking checksum mismatch."
    )


if ranking_hash != p55.get(
    "ranking_source_sha256"
):

    raise RuntimeError(
        "Phase 5.5 source checksum mismatch."
    )


print("\nPhase 5.4 ranking checksum: PASSED")
print("Phase 5.5 provenance correspondence: PASSED")


# ------------------------------------------------------------
# 3. LOAD RESEARCH EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETE RESEARCH EVIDENCE")
print("-" * 65)


ranking = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


evidence = pd.read_csv(
    ALL_TISSUE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


three_way = pd.read_csv(
    THREE_WAY_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


for name, table in {
    "Ranking": ranking,
    "Evidence": evidence
}.items():

    if len(table) != 948:
        raise RuntimeError(
            f"{name}: expected 948 genes."
        )

    if table["gene_id"].isna().any():
        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():
        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


if set(ranking["gene_id"]) != set(
    evidence["gene_id"]
):

    raise RuntimeError(
        "Candidate gene inventories disagree."
    )


print("\nPhase 5.4 candidates:", len(ranking))
print("Phase 5.5 candidates:", len(evidence))

print("\nCandidate inventory correspondence: PASSED")


# ------------------------------------------------------------
# 4. CORRECTED DATA-INTEGRITY COMPARISON
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING CROSS-CSV EVIDENCE INTEGRITY")
print("-" * 65)


# Pandas DataFrame.equals() requires exact
# numerical and dtype agreement.
#
# Repeated CSV serialization can introduce
# negligible floating-point differences.
#
# Our revised audit:
#
# - Aligns genes by original gene_id
# - Requires exact missing-value patterns
# - Requires exact text and boolean agreement
# - Requires exact integer measurement agreement
# - Allows <= 1e-12 absolute/relative tolerance
#   only for continuous numerical measurements
# - Reports every numerical field with differences
# - Raises on any discrepancy outside tolerance


ranking_index = ranking.set_index(
    "gene_id"
)


evidence_index = evidence.set_index(
    "gene_id"
)


original_fields = [
    column
    for column in ranking.columns
    if column != "gene_id"
]


missing_fields = (
    set(original_fields)
    - set(evidence_index.columns)
)


if missing_fields:

    raise RuntimeError(
        "Phase 5.5 is missing original fields: "
        f"{sorted(missing_fields)}"
    )


evidence_aligned = evidence_index.loc[
    ranking_index.index,
    original_fields
]


FLOAT_ATOL = 1e-12
FLOAT_RTOL = 1e-12


numerical_roundtrip_records = []

mismatch_records = []


def is_numeric_series(series):

    return pd.api.types.is_numeric_dtype(
        series
    )


def integer_valued(values):

    finite = values[
        np.isfinite(values)
    ]

    if len(finite) == 0:
        return True

    return bool(
        np.all(
            finite == np.round(finite)
        )
    )


for column in original_fields:

    left = ranking_index[column]
    right = evidence_aligned[column]


    # ----------------------------------------
    # A. Missing-value correspondence
    # ----------------------------------------

    null_equal = (
        left.isna().to_numpy()
        ==
        right.isna().to_numpy()
    )


    if not null_equal.all():

        mismatch_records.append({

            "column": column,
            "reason": "Missing-value pattern changed",
            "n_mismatches": int((~null_equal).sum())
        })

        continue


    # ----------------------------------------
    # B. Numeric field comparison
    # ----------------------------------------

    if (
        is_numeric_series(left)
        and is_numeric_series(right)
    ):

        a = left.to_numpy(
            dtype=float
        )

        b = right.to_numpy(
            dtype=float
        )


        finite = (
            np.isfinite(a)
            &
            np.isfinite(b)
        )


        # Identifier-like numerical columns
        # and integer-valued measurements must
        # agree exactly.
        #
        # This prevents tolerance from masking
        # changes to gene IDs, counts, or ranks.


        identifier_like = (
            "entrez" in column.lower()
            or column.lower().endswith("_id")
        )


        exact_required = (
            identifier_like
            or (
                integer_valued(a)
                and integer_valued(b)
            )
        )


        if exact_required:

            equal = (
                (a == b)
                |
                (
                    np.isnan(a)
                    &
                    np.isnan(b)
                )
            )

        else:

            equal = np.isclose(
                a,
                b,
                rtol=FLOAT_RTOL,
                atol=FLOAT_ATOL,
                equal_nan=True
            )


        if not equal.all():

            bad_indices = np.where(
                ~equal
            )[0]


            mismatch_records.append({

                "column": column,

                "reason": (
                    "Numerical difference exceeds "
                    "allowed precision"
                    if not exact_required
                    else "Exact numerical agreement failed"
                ),

                "n_mismatches": int(
                    len(bad_indices)
                ),

                "example_gene_id": str(
                    ranking_index.index[
                        bad_indices[0]
                    ]
                ),

                "phase54_value": str(
                    a[bad_indices[0]]
                ),

                "phase55_value": str(
                    b[bad_indices[0]]
                )
            })

            continue


        # Record numerical differences that
        # were small enough to be consistent
        # with CSV floating-point round trips.


        if finite.any():

            differences = np.abs(
                a[finite] - b[finite]
            )

            changed = int(
                np.count_nonzero(
                    differences != 0
                )
            )

            if changed > 0:

                numerical_roundtrip_records.append({

                    "column": column,

                    "changed_values": changed,

                    "max_absolute_difference": float(
                        differences.max()
                    ),

                    "allowed_tolerance": (
                        f"atol={FLOAT_ATOL}, "
                        f"rtol={FLOAT_RTOL}"
                    )

                })


    # ----------------------------------------
    # C. Text and categorical comparison
    # ----------------------------------------

    else:

        a = (
            left.astype("string")
            .fillna("<MISSING>")
        )

        b = (
            right.astype("string")
            .fillna("<MISSING>")
        )


        equal = (
            a.to_numpy()
            ==
            b.to_numpy()
        )


        if not equal.all():

            bad_indices = np.where(
                ~equal
            )[0]


            mismatch_records.append({

                "column": column,

                "reason": (
                    "Categorical or textual value changed"
                ),

                "n_mismatches": int(
                    len(bad_indices)
                ),

                "example_gene_id": str(
                    ranking_index.index[
                        bad_indices[0]
                    ]
                ),

                "phase54_value": str(
                    a.iloc[bad_indices[0]]
                ),

                "phase55_value": str(
                    b.iloc[bad_indices[0]]
                )
            })


# ------------------------------------------------------------
# 5. REPORT INTEGRITY AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-CSV INTEGRITY RESULTS")
print("-" * 65)


print("\nOriginal evidence fields checked:",
      len(original_fields))


print(
    "Fields with acceptable numerical round-trip differences:",
    len(numerical_roundtrip_records)
)


print(
    "Fields with substantive discrepancies:",
    len(mismatch_records)
)


if numerical_roundtrip_records:

    roundtrip_df = pd.DataFrame(
        numerical_roundtrip_records
    )


    print("\nAcceptable floating-point differences:")

    print(
        roundtrip_df.sort_values(
            "max_absolute_difference",
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )

else:

    print(
        "\nNo numerical round-trip differences detected."
    )


if mismatch_records:

    mismatch_df = pd.DataFrame(
        mismatch_records
    )

    print("\nSCIENTIFIC DATA DISCREPANCIES DETECTED:")

    print(
        mismatch_df.head(30).to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Phase 5.4 and 5.5 differ beyond "
        "permitted CSV serialization tolerance. "
        "Inspect the printed discrepancies."
    )


print("\nCandidate identifiers: EXACT MATCH")

print("Missing-value patterns: EXACT MATCH")

print("Categorical evidence: EXACT MATCH")

print("Integer-valued measurements: EXACT MATCH")

print("Continuous numerical evidence: VERIFIED")

print("\nOriginal scientific evidence: PRESERVED")


# ------------------------------------------------------------
# 6. VERIFY THREE-SCENARIO AGREEMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING ROBUSTNESS")
print("-" * 65)


SCENARIO_COLUMNS = [
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first"
]


surface = evidence[
    evidence["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS:

    values = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)


    if set(values) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Invalid scenario ranks: {column}"
        )


calculated_three_way = surface[
    surface[SCENARIO_COLUMNS]
    .le(20)
    .all(axis=1)
].copy()


if len(calculated_three_way) != 3:

    raise RuntimeError(
        "Expected three candidates "
        "in all scenario top-20 lists."
    )


if set(
    calculated_three_way["gene_id"]
) != set(
    three_way["gene_id"]
):

    raise RuntimeError(
        "Three-scenario membership changed."
    )


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(
    calculated_three_way["gene_name"]
) != expected_three:

    raise RuntimeError(
        "Unexpected three-scenario candidate identities."
    )


print("\nThree-scenario candidates:")

print(
    calculated_three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS
        ]
    ].to_string(index=False)
)


print("\nScenario rank correlations:")

print(
    correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nCorrelated rankings do not constitute "
    "independent biological validation."
)


# ------------------------------------------------------------
# 7. DEFINE THE RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING BIOLOGICAL RESEARCH PANEL")
print("-" * 65)


REVIEW_GROUPS = {

    "A | Cross-scenario surface hypotheses": [
        "CA9",
        "ADAM12",
        "NPC1L1"
    ],

    "B | Additional surface-antigen hypotheses": [
        "CDH3",
        "CLDN1",
        "DPEP1",
        "SLC7A5",
        "TDGF1",
        "LGR5"
    ],

    "C | Stromal-targeting hypothesis": [
        "FAP"
    ],

    "D | Membrane-accessibility investigations": [
        "SLC5A6",
        "CEMIP"
    ],

    "E | Extracellular microenvironment hypotheses": [
        "CTHRC1",
        "MXRA5",
        "SFRP4"
    ]
}


panel_records = []


for group, genes in REVIEW_GROUPS.items():

    for position, gene in enumerate(
        genes,
        start=1
    ):

        panel_records.append({

            "gene_name": gene,

            "phase56_review_group": group,

            "phase56_group_display_order": position

        })


panel_definition = pd.DataFrame(
    panel_records
)


if len(panel_definition) != 15:

    raise RuntimeError(
        "Expected 15 panel candidates."
    )


if panel_definition["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in panel definition."
    )


if evidence["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate source gene symbols."
    )


panel = panel_definition.merge(
    evidence,
    on="gene_name",
    how="left",
    validate="one_to_one",
    indicator=True
)


if not panel["_merge"].eq("both").all():

    missing = panel.loc[
        panel["_merge"].ne("both"),
        "gene_name"
    ].tolist()

    raise RuntimeError(
        f"Panel genes missing: {missing}"
    )


panel = panel.drop(
    columns="_merge"
)


print("\nResearch-panel genes:", len(panel))


print("\nGroup distribution:")

print(
    panel["phase56_review_group"]
    .value_counts(sort=False)
    .to_string()
)


# ------------------------------------------------------------
# 8. VALIDATE BIOLOGICAL TARGETING LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING TARGETING MECHANISMS")
print("-" * 65)


EXPECTED_LANES = {

    "A": "Putative cell-surface antigen review",

    "B": "Putative cell-surface antigen review",

    "C": "Stromal-associated targeting review",

    "D": (
        "Membrane localization/"
        "accessibility verification"
    ),

    "E": "Extracellular microenvironment review"

}


for _, row in panel.iterrows():

    group_letter = row[
        "phase56_review_group"
    ][0]

    expected_lane = EXPECTED_LANES[
        group_letter
    ]

    actual_lane = row[
        "phase5_followup_lane"
    ]

    if actual_lane != expected_lane:

        raise RuntimeError(
            f"Targeting-lane disagreement "
            f"for {row['gene_name']}: "
            f"{actual_lane}"
        )


print("\nBiological group assignments: PASSED")


# ------------------------------------------------------------
# 9. AUDIT PANEL EVIDENCE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PANEL EVIDENCE")
print("-" * 65)


required_panel_fields = {

    "gene_id",
    "ensembl_stable_id",
    "gene_name",

    "baseline_log2fc",
    "sensitivity_log2fc",
    "phase5_rna_conservative_log2fc",

    "protein_enriched",
    "phase5_protein_evidence_state",
    "mss_complete_pairs",

    "phase5_topology_class",
    "phase5_followup_lane",

    "phase5_normal_ihc_available",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names",

    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",

    *SCENARIO_COLUMNS

}


missing_fields = (
    required_panel_fields
    - set(panel.columns)
)


if missing_fields:

    raise RuntimeError(
        "Missing panel evidence columns: "
        f"{sorted(missing_fields)}"
    )


print("\nPanel evidence fields: VALIDATED")


# ------------------------------------------------------------
# 10. PRESERVE NORMAL-TISSUE MISSINGNESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    invalid = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if invalid.any():

        raise RuntimeError(
            "Unexpected boolean annotations: "
            f"{values[invalid].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    panel["phase5_normal_ihc_available"]
)


normal_count_columns = [
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high"
]


for column in normal_count_columns:

    panel[column] = pd.to_numeric(
        panel[column],
        errors="coerce"
    )


    if panel.loc[
        ~ihc_available,
        column
    ].notna().any():

        raise RuntimeError(
            "Unavailable normal-IHC observations "
            f"were assigned values: {column}"
        )


panel["phase56_normal_ihc_evidence_status"] = (
    np.where(

        ihc_available,

        "HPA normal-IHC observations available",

        "HPA normal-IHC observations unavailable"

    )
)


print(
    "\nPanel genes with normal-IHC observations:",
    int(ihc_available.sum())
)


print(
    "Panel genes without normal-IHC observations:",
    int((~ihc_available).sum())
)


print(
    "\nMissing observations remain missing, "
    "not interpreted as zero expression."
)


# ------------------------------------------------------------
# 11. CREATE EXPERIMENTAL VALIDATION ROADMAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EXPERIMENTAL VALIDATION ROADMAP")
print("-" * 65)


def validation_plan(row):

    gene = row["gene_name"]

    group = row[
        "phase56_review_group"
    ][0]


    if group in ["A", "B"]:

        primary_question = (
            "Is the antigen expressed and "
            "accessible on malignant MSS CRC cells?"
        )

        primary_assay = (
            "Evaluate non-permeabilized flow "
            "cytometry or surface proteomics "
            "in verified MSS CRC models"
        )

        orthogonal_assay = (
            "Use single-cell/spatial expression "
            "analysis and matched normal-colon "
            "organoids to evaluate selectivity"
        )


    elif group == "C":

        primary_question = (
            "Is the protein enriched in "
            "tumor-associated fibroblasts?"
        )

        primary_assay = (
            "Perform multiplex immunostaining "
            "with epithelial and fibroblast markers"
        )

        orthogonal_assay = (
            "Compare tumor-stromal and normal-tissue "
            "localization using spatial protein assays"
        )


    elif group == "D":

        primary_question = (
            "Is an extracellular epitope accessible "
            "on intact cancer cells?"
        )

        primary_assay = (
            "Evaluate live-cell extracellular "
            "antibody binding and membrane localization"
        )

        orthogonal_assay = (
            "Verify surface topology using "
            "orthogonal surface-proteomics methods"
        )


    else:

        primary_question = (
            "Where is the extracellular protein "
            "localized in the tumor environment?"
        )

        primary_assay = (
            "Characterize secreted or extracellular "
            "protein distribution in tumor sections"
        )

        orthogonal_assay = (
            "Compare tumor versus normal-tissue "
            "localization with spatial protein assays"
        )


    additional = []


    protein_state = str(
        row["phase5_protein_evidence_state"]
    )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        additional.append(
            "Obtain stronger tumor/normal "
            "protein-level evidence"
        )


    if (
        row["phase56_normal_ihc_evidence_status"]
        ==
        "HPA normal-IHC observations unavailable"
    ):

        additional.append(
            "Acquire independent normal-tissue "
            "expression measurements"
        )


    if gene in ["DPEP1", "SLC7A5"]:

        additional.append(
            "Investigate normal colon and kidney "
            "cell-surface expression"
        )


    if gene in ["CDH3", "CLDN1"]:

        additional.append(
            "Improve tumor/normal protein coverage "
            "and investigate normal epithelial expression"
        )


    if gene == "TDGF1":

        additional.append(
            "Validate GPI-associated surface "
            "presentation and potential shedding"
        )


    if gene == "FAP":

        additional.append(
            "Distinguish tumor-associated stromal "
            "expression from malignant-cell expression"
        )


    return {

        "primary_question": primary_question,

        "primary_assay": primary_assay,

        "orthogonal_assay": orthogonal_assay,

        "additional_review": (
            "; ".join(additional)
            if additional
            else "Standard specificity and validation review"
        )

    }


validation_records = []


for _, row in panel.iterrows():

    plan = validation_plan(row)

    validation_records.append({

        "gene_id": row["gene_id"],

        "gene_name": row["gene_name"],

        "review_group":
            row["phase56_review_group"],

        **plan

    })


validation = pd.DataFrame(
    validation_records
)


if len(validation) != 15:

    raise RuntimeError(
        "Validation roadmap lost candidates."
    )


print("\nCandidate-specific validation plans:", len(validation))

print("Experimental validation roadmap: PREPARED")


# ------------------------------------------------------------
# 12. DISPLAY RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING RESEARCH PANEL EVIDENCE")
print("-" * 65)


panel_display_columns = [

    "gene_name",

    "phase56_review_group",

    "phase5_rna_conservative_log2fc",

    "phase5_protein_evidence_state",

    "phase5_topology_class",

    "phase55_normal_tissues_medium_high",

    "depmap_crc_median_gene_effect"

]


print("\n")

print(
    panel[
        panel_display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nPanel order reflects review groups, "
      "not therapeutic superiority.")


# ------------------------------------------------------------
# 13. PRESERVE COMPLETE CANDIDATE UNIVERSE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING COMPLETE 948-GENE UNIVERSE")
print("-" * 65)


panel_lookup = (
    panel.set_index("gene_id")[
        "phase56_review_group"
    ].to_dict()
)


full_audit = evidence.copy(deep=True)


full_audit["phase56_in_review_panel"] = (
    full_audit["gene_id"].isin(
        set(panel["gene_id"])
    )
)


full_audit["phase56_review_group"] = (
    full_audit["gene_id"]
    .map(panel_lookup)
)


if len(full_audit) != 948:

    raise RuntimeError(
        "Original research universe changed."
    )


if int(
    full_audit["phase56_in_review_panel"]
    .sum()
) != 15:

    raise RuntimeError(
        "Panel membership count incorrect."
    )


if not evidence.equals(
    full_audit[evidence.columns]
):

    raise RuntimeError(
        "Original research evidence changed."
    )


print("\nAll candidate genes preserved:", len(full_audit))

print("Detailed research-panel genes:", 15)

print("Other retained candidates:", 933)


# ------------------------------------------------------------
# 14. PREPARE NORMAL-TISSUE REVIEW TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE REVIEW")
print("-" * 65)


normal_review_columns = [

    "gene_id",
    "gene_name",
    "phase56_review_group",

    "phase56_normal_ihc_evidence_status",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names"

]


normal_review = panel[
    normal_review_columns
].copy()


print("\nNormal-tissue review: PREPARED")


# ------------------------------------------------------------
# 15. SAVE PANEL TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL RESULTS")
print("-" * 65)


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

VALIDATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

NORMAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_panel_normal_tissue_review.csv"
)

INTEGRITY_PATH = (
    META /
    "surfacetarget_phase5_roundtrip_integrity_audit.json"
)


panel.to_csv(
    PANEL_PATH,
    index=False
)


full_audit.to_csv(
    FULL_PATH,
    index=False
)


validation.to_csv(
    VALIDATION_PATH,
    index=False
)


normal_review.to_csv(
    NORMAL_PATH,
    index=False
)


integrity_report = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6A",

    "comparison": (
        "Phase 5.4 ranking CSV versus "
        "Phase 5.5 all-tissue evidence CSV"
    ),

    "candidate_genes": 948,

    "fields_checked": len(original_fields),

    "numerical_tolerance": {
        "absolute": FLOAT_ATOL,
        "relative": FLOAT_RTOL
    },

    "accepted_numerical_roundtrip_fields":
        numerical_roundtrip_records,

    "substantive_discrepancies":
        mismatch_records,

    "integrity_passed": True,

    "ranking_source_sha256": ranking_hash,

    "evidence_source_sha256":
        sha256_file(ALL_TISSUE_PATH)

}


with open(
    INTEGRITY_PATH,
    "w"
) as handle:

    json.dump(
        integrity_report,
        handle,
        indent=4
    )


saved_panel = pd.read_csv(
    PANEL_PATH,
    dtype={"gene_id": str}
)


saved_full = pd.read_csv(
    FULL_PATH,
    dtype={"gene_id": str}
)


if len(saved_panel) != 15:

    raise RuntimeError(
        "Saved research panel count incorrect."
    )


if len(saved_full) != 948:

    raise RuntimeError(
        "Saved candidate universe count incorrect."
    )


if set(saved_panel["gene_id"]) != set(
    panel["gene_id"]
):

    raise RuntimeError(
        "Saved research panel identifiers changed."
    )


print("\nPanel tables: SAVED")

print("Integrity audit: SAVED")

print("Output validation: PASSED")


# ------------------------------------------------------------
# 16. CREATE HUMAN-READABLE RESEARCH REPORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH INTERPRETATION REPORT")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase5_Research_Panel.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 5 — Research Panel and Validation Roadmap",

    "",

    (
        "This report summarizes 15 exploratory research "
        "candidates selected from 948 colorectal "
        "tumor-associated transcriptomic candidates."
    ),

    "",

    (
        "The panel combines cross-scenario ranking "
        "agreement with complementary mechanistic "
        "research hypotheses."
    ),

    "",

    (
        "It does not represent a validated therapeutic "
        "ranking, safety assessment, or clinical recommendation."
    ),

    "",

    "## Research evidence",

    "",

    (
        "- TCGA: tumor-associated RNA evidence, "
        "including computationally low-MSI discovery cases."
    ),

    (
        "- CPTAC: paired MSS colon tumor/normal "
        "proteomics, subject to protein-specific coverage."
    ),

    (
        "- UniProt/HPA: topology and descriptive "
        "normal-tissue expression annotations."
    ),

    (
        "- DepMap: colorectal adenocarcinoma CRISPR "
        "fitness measurements without confirmed MSS "
        "status in this analysis."
    ),

    "",

    "## Ranking limitations",

    "",

    (
        "The protein-first and normal-review-first "
        "rankings are highly correlated."
    ),

    (
        "Only CA9, ADAM12, and NPC1L1 appeared "
        "in the top 20 under all three scenarios."
    ),

    (
        "Cross-scenario agreement does not constitute "
        "independent biological validation."
    ),

    "",

    "## Candidate research panel",

    ""
]


for group, subset in panel.groupby(
    "phase56_review_group",
    sort=False
):

    report_lines.extend([

        f"### {group}",

        ""

    ])


    for _, row in subset.iterrows():

        gene = row["gene_name"]

        rna = row[
            "phase5_rna_conservative_log2fc"
        ]

        protein_state = row[
            "phase5_protein_evidence_state"
        ]

        topology = row[
            "phase5_topology_class"
        ]

        normal_n = row[
            "phase55_normal_tissues_medium_high"
        ]

        effect = row[
            "depmap_crc_median_gene_effect"
        ]


        normal_text = (
            "unavailable"
            if pd.isna(normal_n)
            else str(int(normal_n))
        )


        effect_text = (
            "unavailable"
            if pd.isna(effect)
            else f"{effect:.3f}"
        )


        plan = validation.loc[
            validation["gene_name"].eq(gene)
        ].iloc[0]


        report_lines.extend([

            f"**{gene}**",

            "",

            (
                f"- Conservative RNA log2FC: "
                f"{rna:.3f}"
            ),

            f"- CPTAC evidence: {protein_state}",

            f"- Surface topology: {topology}",

            (
                "- Normal tissues with medium/high "
                f"IHC: {normal_text}"
            ),

            (
                "- Median colorectal CRISPR "
                f"gene effect: {effect_text}"
            ),

            (
                "- Primary validation question: "
                f"{plan['primary_question']}"
            ),

            (
                "- Suggested first assay: "
                f"{plan['primary_assay']}"
            ),

            ""
        ])


report_lines.extend([

    "## Scientific limitations",

    "",

    (
        "Bulk tumor RNA and proteomics do not "
        "establish malignant-cell-specific expression."
    ),

    "",

    (
        "UniProt topology does not directly demonstrate "
        "antibody accessibility or internalization."
    ),

    "",

    (
        "HPA staining does not establish clinical "
        "safety or predict therapeutic toxicity."
    ),

    "",

    (
        "CRISPR knockout effects do not directly "
        "measure antibody-drug conjugate efficacy."
    ),

    "",

    (
        "Missing evidence has not been interpreted "
        "as negative biological evidence."
    ),

    "",

    (
        "All 948 original research candidates "
        "remain preserved in the complete dataset."
    ),

    "",

    (
        "No targets have been experimentally "
        "validated by this project."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nResearch report created.")

print("Location:", REPORT_PATH)


# ------------------------------------------------------------
# 17. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL PROVENANCE")
print("-" * 65)


PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)


panel_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6A",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)

        for name, path in required_files.items()

    },

    "integrity_validation": (
        "Exact gene, missingness, categorical "
        "and integer correspondence; strict "
        "floating-point tolerance for continuous "
        "CSV round-trip measurements"
    ),

    "integrity_audit": str(
        INTEGRITY_PATH
    ),

    "candidate_universe": 948,

    "research_panel_size": 15,

    "panel_selection": REVIEW_GROUPS,

    "cross_scenario_consistent_genes": sorted(
        expected_three
    ),

    "panel_method": (
        "Cross-scenario ranking agreement "
        "plus explicitly selected complementary "
        "biological research hypotheses"
    ),

    "ranking_scenario_independence": False,

    "therapeutic_scores_assigned": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "all_candidate_genes_preserved": True,

    "panel_output": str(PANEL_PATH),

    "panel_sha256": sha256_file(PANEL_PATH),

    "full_audit_output": str(FULL_PATH),

    "full_audit_sha256": sha256_file(FULL_PATH),

    "report_output": str(REPORT_PATH),

    "report_sha256": sha256_file(REPORT_PATH),

    "limitations": [

        "Panel selection is a research review "
        "decision, not a clinically validated ranking.",

        "Two prioritization scenarios are "
        "strongly correlated.",

        "RNA discovery includes computationally "
        "low-MSI rather than exclusively "
        "independently confirmed MSS cases.",

        "DepMap colorectal models do not have "
        "independently verified MSS status here.",

        "Protein-level measurement coverage "
        "varies substantially across genes.",

        "Bulk molecular measurements cannot "
        "establish malignant-cell origin.",

        "Normal-tissue IHC is descriptive "
        "and cannot establish clinical safety.",

        "Cell-surface topology does not prove "
        "antibody binding or accessibility.",

        "No new experimental target "
        "validation was performed."

    ]
}


with open(
    PANEL_PROVENANCE_PATH,
    "w"
) as handle:

    json.dump(
        panel_provenance,
        handle,
        indent=4
    )


print("\nResearch provenance: SAVED")


# ------------------------------------------------------------
# 18. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch56A_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if not source_folder.exists():
            continue

        for file_path in source_folder.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_size = (
    checkpoint.stat().st_size
    / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.6A SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(full_audit))

print("Detailed research panel:", len(panel))

print(
    "Cross-scenario-consistent surface candidates:",
    len(calculated_three_way)
)

print(
    "Biological research groups:",
    len(REVIEW_GROUPS)
)


print("\nScientific integrity audit: PASSED")

print("Source evidence: PRESERVED")

print("Three-scenario agreement: VERIFIED")

print("Normal-tissue missingness: PRESERVED")


print("\nSaved outputs:")

print("1. surfacetarget_phase5_research_panel_v1.csv")

print("2. surfacetarget_phase5_full_universe_panel_audit.csv")

print("3. surfacetarget_phase5_experimental_validation_roadmap.csv")

print("4. surfacetarget_phase5_panel_normal_tissue_review.csv")

print("5. SurfaceTarget_CRC_Phase5_Research_Panel.md")

print("6. surfacetarget_phase5_roundtrip_integrity_audit.json")

print("7. surfacetarget_phase5_research_panel_provenance.json")


print("\nCheckpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidates excluded.")

print("No clinical safety scores assigned.")

print("No therapeutic efficacy claims.")

print("No experimental validation claimed.")


print("\nPATCH 5.6A COMPLETE")

print("=" * 65)